# Generating shrunken logFC (posterior-mean) matrices

Turns a perturb-seq screen into a shrunken log-fold-change matrix in which **each
perturbation appears several times** — once from all its cells, and once per subsample of
them. Useful when you want many logFC estimates per perturbation at varying precision.

Pipeline (scripts under `src/module_finder/de/`):

| step | script | what it does |
|---|---|---|
| 1 | `ComputeSE.py` | cells → per-label `mean_diff` + Welch `se`, chunked |
| 2 | `run_ashr_on_chunk.R` | `ash(mean_diff, se)` per chunk |
| 3 | `MergeAshrRes.py` | merge label/gene metadata back in |
| 4 | `AshrGenerateMatrix.py` | pivot to wide label × gene `PosteriorMean` |

**Subsampling.** Perturbations with more than 50 cells get `N_SUBSAMPLES` subsamples whose
sizes span `[50, n_cells)`; the cells in each are drawn uniformly without replacement.
Perturbations at or below 50 cells contribute their full row only.

**Controls.** A fixed set of `N_CONTROL_CELLS` controls is drawn once and shared by every
row, so the control side of the standard error is identical across rows.

Every knob is a plain variable in the config cell below, passed straight to the
functions — edit the cell to change behaviour. Note the two different floors:
`MIN_CELLS_TO_SUBSAMPLE` (50) decides which perturbations are *eligible for subsampling*,
while `MIN_CELLS` (20) is `ComputeSE.py`'s cutoff for dropping a perturbation
altogether.

Requires R with the `ashr` package.

In [ ]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parents[1]          # notebooks/module_finder -> project root
sys.path.insert(0, str(PROJECT / "src"))

import numpy as np
import pandas as pd
import anndata as ad

from module_finder.de import (
    generate_posterior_matrices,
    plan_augmentation,
    build_augmented_adata,
    subsample_sizes,
)

## 1. Point at your data

You need an AnnData with:

- log-normalised expression in `.X` (or name a `layer`),
- a `.obs` column of perturbation labels,
- a control label within that column.

`CONTEXT` is just a name for this dataset — it names the output files. Run one call per
condition if your screen has several.

In [ ]:
ADATA_PATH = Path("/path/to/your/screen.h5ad")
CONTEXT = "my_screen"

PERTURBATION_KEY = "target_gene"
CONTROL_LABEL = "non-targeting"
LAYER = None                    # None uses .X

# ---- user-defined ----
N_SUBSAMPLES = 6                # subsamples per eligible perturbation
SPACING = "linear"              # "log" covers precision more evenly (se ~ 1/sqrt(n))
MIN_CELLS_TO_SUBSAMPLE = 50     # only subsample perturbations with more cells than this
N_CONTROL_CELLS = 10_000        # fixed control set, shared by every row
MIN_CELLS = 20                  # ComputeSE.py drops perturbations below this entirely

OUT_DIR = PROJECT / "data" / "posterior_matrices"
OUT_DIR.mkdir(parents=True, exist_ok=True)

adata = ad.read_h5ad(ADATA_PATH, backed="r")
counts = adata.obs[PERTURBATION_KEY].value_counts()

print(adata)
print(f"\nperturbations : {len(counts) - 1}")
print(f"control cells : {counts.get(CONTROL_LABEL, 0):,}")
print(f"cells/pert    : min {counts.drop(CONTROL_LABEL, errors='ignore').min()}, "
      f"median {int(counts.drop(CONTROL_LABEL, errors='ignore').median())}, "
      f"max {counts.drop(CONTROL_LABEL, errors='ignore').max():,}")
print(f"eligible for subsampling (>{MIN_CELLS_TO_SUBSAMPLE} cells): "
      f"{int((counts.drop(CONTROL_LABEL, errors='ignore') > MIN_CELLS_TO_SUBSAMPLE).sum())}")

In [ ]:
# Optional: work on a subset first. `ComputeSE.py` loads the object into memory, so on a
# large screen either subset here or run on shards.
SUBSET_PERTS = 40      # None to use everything
SUBSET_GENES = 1500    # None for all genes

if SUBSET_PERTS is not None:
    keep = [CONTROL_LABEL] + [p for p in counts.index if p != CONTROL_LABEL][:SUBSET_PERTS]
    mask = adata.obs[PERTURBATION_KEY].isin(keep).to_numpy()
    genes = slice(None) if SUBSET_GENES is None else slice(0, SUBSET_GENES)
    adata = adata[mask, genes].to_memory()
    print(adata.shape, "|", adata.obs[PERTURBATION_KEY].nunique(), "labels")

## 2. Check the subsample sizes

`subsample_sizes` is pure arithmetic — see what you will get before touching the data.
The full row sits at `n_cells`, so the subsamples stop just below it and together they
span the range.

In [ ]:
for n_cells in (60, 300, 2000, 10000):
    print(f"{n_cells:>6} cells -> "
          f"{subsample_sizes(n_cells, N_SUBSAMPLES, MIN_CELLS_TO_SUBSAMPLE, SPACING)}"
          f" + full at {n_cells}")

## 3. Inspect the plan

`plan_augmentation` decides every output row before any expression is read, so the rule
is auditable on its own.

In [ ]:
perturbations = adata.obs[PERTURBATION_KEY].astype(str).to_numpy()

cell_index, labels, plan = plan_augmentation(
    perturbations,
    control_label=CONTROL_LABEL,
    n_subsamples=N_SUBSAMPLES,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    spacing=SPACING,
    seed=0,
)

print(f"{len(plan)} planned rows from {adata.n_obs:,} cells "
      f"-> {len(cell_index):,} cells after duplication\n")
print(plan["variant"].value_counts().to_string())
plan.head(12)

In [ ]:
full = plan[plan.variant == "full"].set_index("perturbation")["n_cells_planned"]
sub = plan[plan.variant == "subsample"]
too_small = full[full <= MIN_CELLS_TO_SUBSAMPLE].index

print("perturbations at/below the floor:", len(too_small))
print("  none subsampled              :", not sub.perturbation.isin(too_small).any())
print("  every n >= floor             :", bool((sub.n_cells_planned >= MIN_CELLS_TO_SUBSAMPLE).all()))
print("  no subsample equals full size:",
      bool((sub.n_cells_planned < full.reindex(sub.perturbation).to_numpy()).all()))
print("  control set fixed at         :",
      int(plan.loc[plan.variant == "control", "n_cells_planned"].item()), "cells")

per_pert = sub.groupby("perturbation").n_cells_planned.agg(["count", "min", "max"])
print(f"\nsubsamples per perturbation: median {int(per_pert['count'].median())} "
      f"of {N_SUBSAMPLES} requested")
per_pert.head()

## 4. Build the augmented object

Full-data cells keep their perturbation label; subsampled cells are **duplicated** under
`<pert>__sub<k>`, which is what lets one `ComputeSE.py` pass produce both variants — and
puts them in one `ashr` fit, where their differing precisions belong. Controls are never
duplicated.

This cell is optional: step 5 does it for you. Run it to see the size cost.

In [ ]:
augmented, plan = build_augmented_adata(
    adata,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    n_subsamples=N_SUBSAMPLES,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    spacing=SPACING,
    seed=0,
)
print(f"{adata.n_obs:,} cells -> {augmented.n_obs:,} ({augmented.n_obs / adata.n_obs:.2f}x)")
augmented.obs["mf_label"].value_counts().head(8)

## 5. Run it

One call covers the whole chain: build the augmented object, write it, run all four
scripts, collect the results.

`chunk_perts` sets how many labels share an `ashr` fit. Raise it above the row count if
you want a single prior over every row.

In [ ]:
run = generate_posterior_matrices(
    adata,
    context=CONTEXT,
    out_dir=OUT_DIR,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    layer=LAYER,
    n_subsamples=N_SUBSAMPLES,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    spacing=SPACING,
    seed=0,
    min_cells=MIN_CELLS,
    chunk_perts=100,
    n_ashr_jobs=6,
    matrix_format="csv",
)
print(run)
print("\nmatrix       :", run.matrix_path)
print("row metadata :", run.row_metadata_path)
print("ashr version :", run.meta["ashr_version"])
print("build %.1fs, pipeline %.1fs" % (run.meta["build_seconds"], run.meta["pipeline_seconds"]))

## 6. Results

`run.matrix` is labels × genes: `<pert>` rows are full-data estimates, `<pert>__sub<k>`
rows are subsamples. `run.row_metadata` says how many cells produced each row.

In [ ]:
print(run.matrix.shape)
run.matrix.iloc[:6, :5]

In [ ]:
run.row_metadata.head(10)

In [ ]:
meta = run.row_metadata.set_index("label")
subs = meta[meta.variant == "subsample"]

rows = []
for label, r in subs.iterrows():
    if r.perturbation in run.matrix.index:
        a, b = run.matrix.loc[r.perturbation], run.matrix.loc[label]
        rows.append({
            "perturbation": r.perturbation,
            "n_sub": int(r.n_cells_used),
            "frac_cells": r.n_cells_used / meta.loc[r.perturbation, "n_cells_used"],
            "corr_to_full": np.corrcoef(a, b)[0, 1],
            "shrinkage_ratio": np.abs(b).mean() / max(np.abs(a).mean(), 1e-12),
        })
concordance = pd.DataFrame(rows)
print(concordance[["frac_cells", "corr_to_full", "shrinkage_ratio"]].describe().round(3).to_string())
concordance.head()

In [ ]:
# fewer cells -> larger se -> ashr shrinks harder
ax = concordance.plot.scatter("frac_cells", "shrinkage_ratio", alpha=.6,
                              title="subsampled rows are shrunk more")
ax.set_xlabel("fraction of cells used"); ax.set_ylabel("|PosteriorMean| relative to full");

## 7. A label-permuted null (optional)

Same call with `permute=True`: perturbation labels are shuffled among the perturbed
cells, controls untouched, every label keeping its cell count.

It is a null for **specificity**, not for overall effect size — the average perturbation
effect still reaches every row, because the control baseline is left intact.

In [ ]:
null_run = generate_posterior_matrices(
    adata,
    context=CONTEXT,
    out_dir=OUT_DIR,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    layer=LAYER,
    n_subsamples=N_SUBSAMPLES,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    spacing=SPACING,
    seed=0,
    permute=True,
    permute_seed=0,
    min_cells=MIN_CELLS,
    chunk_perts=100,
    n_ashr_jobs=6,
)
print(null_run)

real, null = run.matrix.std(axis=0).mean(), null_run.matrix.std(axis=0).mean()
print(f"\nmean across-row SD per gene: real {real:.5f} | permuted {null:.5f} "
      f"| ratio {real / max(null, 1e-12):.2f}x")
print("(near 1 means no more perturbation-specific structure than the null)")

## 8. What landed on disk

```
<OUT_DIR>/<CONTEXT>/
    PosteriorMean_matrix_<CONTEXT>.csv   labels x genes, full + subsampled rows
    row_metadata.csv                     label -> perturbation, variant, n_cells_used
    module_finder_manifest.json          parameters, timings, ashr version
    chunk_*.csv                          ComputeSE.py output (mean_diff, se)
    AshrResult_chunk_*.csv               ashr output, metadata merged back in
```

Intermediates are kept, so a re-run skips `ComputeSE.py`. Delete `chunk_*.csv` to force
a recompute.

In [ ]:
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file() and not path.name.startswith("chunk"):
        print(f"{path.stat().st_size/1e6:9.2f} MB  {path.relative_to(OUT_DIR)}")